<a href="https://colab.research.google.com/github/aviralraghav/data-analyst-assessment-olist/blob/main/Virtubox_DataAnalyst_assessment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import pandas as pd
import numpy as np

customers = pd.read_csv("olist_customers_dataset.csv")
geolocation = pd.read_csv("olist_geolocation_dataset.csv")
order_items = pd.read_csv("olist_order_items_dataset.csv")
payments = pd.read_csv("olist_order_payments_dataset.csv")
reviews = pd.read_csv("olist_order_reviews_dataset.csv")
orders = pd.read_csv("olist_orders_dataset.csv")
products = pd.read_csv("olist_products_dataset.csv")
sellers = pd.read_csv("olist_sellers_dataset.csv")
translation = pd.read_csv("product_category_name_translation.csv")

print("All 9 datasets loaded successfully!")

All 9 datasets loaded successfully!


In [4]:
datasets = {
    "customers": customers,
    "geolocation": geolocation,
    "order_items": order_items,
    "payments": payments,
    "reviews": reviews,
    "orders": orders,
    "products": products,
    "sellers": sellers,
    "translation": translation
}

for name, df in datasets.items():
    print(f"{name}: {df.shape[0]:,} rows × {df.shape[1]} columns")

customers: 99,441 rows × 5 columns
geolocation: 1,000,163 rows × 5 columns
order_items: 112,650 rows × 7 columns
payments: 103,886 rows × 5 columns
reviews: 99,224 rows × 7 columns
orders: 99,441 rows × 8 columns
products: 32,951 rows × 9 columns
sellers: 3,095 rows × 4 columns
translation: 71 rows × 2 columns


In [5]:
for name, df in datasets.items():
    print(f"\n===== {name.upper()} =====")
    print("Duplicate rows:", df.duplicated().sum())
    print("Missing values:")
    print(df.isna().sum()[df.isna().sum() > 0])


===== CUSTOMERS =====
Duplicate rows: 0
Missing values:
Series([], dtype: int64)

===== GEOLOCATION =====
Duplicate rows: 261831
Missing values:
Series([], dtype: int64)

===== ORDER_ITEMS =====
Duplicate rows: 0
Missing values:
Series([], dtype: int64)

===== PAYMENTS =====
Duplicate rows: 0
Missing values:
Series([], dtype: int64)

===== REVIEWS =====
Duplicate rows: 0
Missing values:
review_comment_title      87656
review_comment_message    58247
dtype: int64

===== ORDERS =====
Duplicate rows: 0
Missing values:
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
dtype: int64

===== PRODUCTS =====
Duplicate rows: 0
Missing values:
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64

===== SEL

In [6]:
# Q3 – Data cleaning and preparation

# 1. Remove exact duplicate rows
for name, df in datasets.items():
    datasets[name] = df.drop_duplicates().copy()

# 2. Convert date columns
date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in date_cols:
    if col in orders.columns:
        orders[col] = pd.to_datetime(orders[col], errors="coerce")

# 3. Create analysis-ready transaction dataset
analysis_df = (
    order_items
    .merge(orders, on="order_id", how="left")
    .merge(products, on="product_id", how="left")
    .merge(sellers, on="seller_id", how="left")
    .merge(translation, on="product_category_name", how="left")
)

# 4. Aggregate payments at order level
payment_summary = (
    payments.groupby("order_id", as_index=False)
    .agg(
        total_payment_value=("payment_value", "sum"),
        max_installments=("payment_installments", "max")
    )
)

analysis_df = analysis_df.merge(payment_summary, on="order_id", how="left")

# 5. Aggregate reviews at order level
review_summary = (
    reviews.groupby("order_id", as_index=False)
    .agg(
        review_score=("review_score", "mean"),
        review_count=("review_id", "count")
    )
)

analysis_df = analysis_df.merge(review_summary, on="order_id", how="left")

# 6. Calculated fields
analysis_df["item_revenue"] = (
    analysis_df["price"].fillna(0) +
    analysis_df["freight_value"].fillna(0)
)

analysis_df["delivery_days"] = (
    analysis_df["order_delivered_customer_date"]
    - analysis_df["order_purchase_timestamp"]
).dt.days

analysis_df["delivery_delay_days"] = (
    analysis_df["order_delivered_customer_date"]
    - analysis_df["order_estimated_delivery_date"]
).dt.days

analysis_df["delivery_status"] = np.where(
    analysis_df["delivery_delay_days"] > 0,
    "Late",
    "On Time"
)

# 7. Standardize missing category values
if "product_category_name_english" in analysis_df.columns:
    analysis_df["product_category_name_english"] = (
        analysis_df["product_category_name_english"].fillna("Unknown")
    )

print("Processed dataset shape:", analysis_df.shape)
print("Processing completed successfully!")

Processed dataset shape: (112650, 34)
Processing completed successfully!


In [7]:
# Save processed data
analysis_df.to_csv("processed_olist_analysis.csv", index=False)

print("Saved: processed_olist_analysis.csv")

Saved: processed_olist_analysis.csv
